In [5]:

!pip install ortools pandas numpy --break-system-packages
import pandas as pd
import numpy as np
import time
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# --- 1. Load Data (Identical to Assignment 1) ---
CITIES_FILE = "india_cities_1000.csv"
DISTANCES_FILE = "distance_matrix.csv"

print("Loading data...")
cities_df = pd.read_csv(CITIES_FILE)
distances_df = pd.read_csv(DISTANCES_FILE)

# Create mapping dictionary from City Name to its Index
city_list = cities_df['Place_Name'].tolist()
city_to_idx = {name: idx for idx, name in enumerate(city_list)}

# Initialize empty 1000x1000 matrix
full_dist_matrix = np.zeros((1000, 1000))

# Map string names to integer coordinates
distances_df['from_idx'] = distances_df['fromplace'].map(city_to_idx)
distances_df['to_idx'] = distances_df['toplace'].map(city_to_idx)

# Populate the matrix
full_dist_matrix[distances_df['from_idx'], distances_df['to_idx']] = distances_df['dist_km']

# Instance sizes requested for the assignment
N_SIZES = [10, 50, 100, 250, 500, 1000]

# --- 2. OR-Tools Setup & Solver Function ---
def solve_tsp_ortools(n):
    print(f"Solving for N = {n} with OR-Tools...")
    
    # Slice the data for the current number of cities
    dist_matrix = full_dist_matrix[:n, :n]
    
    # Step A: Initialize the Routing Index Manager
    # Parameters: (number of cities, number of vehicles, depot index)
    manager = pywrapcp.RoutingIndexManager(n, 1, 0)
    
    # Step B: Create the Routing Model
    routing = pywrapcp.RoutingModel(manager)
    
    # Step C: Define the Distance Callback
    # OR-Tools requires integer costs. We convert our km (floats) to meters (integers).
    def distance_callback(from_index, to_index):
        # Convert internal routing indices to our matrix node indices
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        # Multiply km by 1000 to get integer meters
        return int(dist_matrix[from_node, to_node] * 1000)

    # Register the callback with the routing model
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    
    # Tell OR-Tools to use this callback as the cost it needs to minimize
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    
    # Step D: Configure Search Parameters (The Heuristic Strategy)
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    
    # 1. First-Solution Strategy: Connect closest arcs first to get a fast baseline
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    
    # 2. Metaheuristic: Use Guided Local Search to swap edges and improve the route
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
    
    # 3. Hard Time Limit: 30 seconds per instance
    search_parameters.time_limit.seconds = 30
    
    # Step E: Solve!
    start_time = time.time()
    solution = routing.SolveWithParameters(search_parameters)
    runtime = time.time() - start_time
    
    # Step F: Extract and Format Results
    # OR-Tools status codes: 1 = Success, 2 = Partial Success (Heuristic hit time limit but found a route)
    status_code = routing.status()
    status_map = {
        0: "Not Called",
        1: "Optimal/Success", 
        2: "Heuristic (Time Limit)", # Typical for Guided Local Search when hitting the 30s cap
        3: "Fail",
        4: "Timeout (No Solution)"
    }
    status = status_map.get(status_code, "Unknown")
    
    if solution:
        # The objective is in meters. Divide by 1000 to get it back to km for the table.
        total_distance_km = solution.ObjectiveValue() / 1000.0
        
        return {
            "N (cities)": n,
            "Total Distance (km)": f"{total_distance_km:.2f}",
            "Runtime (s)": f"{runtime:.2f}",
            "Solver Status": status,
            "Notes": "PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH (30s limit)"
        }
    else:
        return {
            "N (cities)": n,
            "Total Distance (km)": "N/A",
            "Runtime (s)": f"{runtime:.2f}",
            "Solver Status": status,
            "Notes": "Failed to find a solution"
        }

# --- 3. Execution Block ---
if __name__ == "__main__":
    results = []
    
    for n in N_SIZES:
        res = solve_tsp_ortools(n)
        results.append(res)
        
    # Print Table 1 exactly as requested in the assignment brief
    print("\n" + "="*95)
    print("TABLE 1 - GOOGLE OR-TOOLS (HEURISTIC)")
    print("="*95)
    final_df = pd.DataFrame(results)
    print(final_df.to_string(index=False))
    
    # Save to CSV for easy copy-pasting into your report
    final_df.to_csv("TSP_Task1_Results.csv", index=False)

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 56.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ortools]m3/4 [ortools]
Loading data...
Solving for N = 10 with OR-Tools...
Solving for N = 50 with OR-Tools...
Solving for N = 100 with OR-Tools...
Solving for N = 250 with OR-Tools...
Solving for N = 500 with OR-Tools...
Solving for N = 1000 with OR-Tools...

TABLE 1 - GOOGLE OR-TOOLS (HEURISTIC)
 N (cities) Total Distance (km) Runtime (s)          Solver Status                                               Notes
         10             6176.00       30.00        Optimal/Success PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH (30s limit)
         50            11745.00       30.00        Optimal/Success PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH (30s limit)
        100            16979.00       30.00        Optimal/Success PATH_CHEAPEST_ARC + GUIDED_LOCAL_SEARCH (30s limit)
        

In [1]:
pip show ortools

Name: ortools
Version: 9.15.6755
Summary: Google OR-Tools python libraries and modules
Home-page: https://developers.google.com/optimization/
Author: Google LLC
Author-email: or-tools@google.com
License: Apache 2.0
Location: <HOME>/.local/lib/python3.13/site-packages
Requires: absl-py, immutabledict, numpy, pandas, protobuf, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ortools
print(ortools.__version__)

9.15.6755
